# Nebula X Hackathon: AI Pipeline (Problem Statement 3)
**Team:** Ren, May, Thuzar
**Goal:** Ingest telemetry, detect anomalies, classify faults, and forecast threshold breaches.

## 0. Setup & Global Imports
Run this first to load all required libraries.

In [ ]:
import os
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# Optional: Set random seed for reproducibility
np.random.seed(42)

## 1. Data Ingestion
Loads local CSV files into memory.

In [ ]:
def load_local_csvs(data_dir: str) -> dict:
    """
    Loads local CSV files into a dictionary for the pipeline.
    """
    print(f"Loading CSVs from local directory: {data_dir}")
    
    bogie_df = pd.read_csv(os.path.join(data_dir, 'bogie_data.csv'))
    doors_df = pd.read_csv(os.path.join(data_dir, 'door_data.csv'))
    faults_df = pd.read_csv(os.path.join(data_dir, 'manually_verified_fault_data.csv'))
    
    return {'bogie': bogie_df, 'doors': doors_df, 'faults': faults_df}

## 2. Preprocessing & Data Resilience
Merges multiple datasets asynchronously, splits them chronologically, and engineers rolling features.

In [ ]:
def merge_telemetry_datasets(telemetry_files: dict, time_col: str, id_col: str) -> pd.DataFrame:
    dataframes = []
    for name, df in telemetry_files.items():
        df[time_col] = pd.to_datetime(df[time_col])
        df = df.sort_values(by=time_col)
        cols_to_rename = {col: f"{name}_{col}" for col in df.columns if col not in [time_col, id_col]}
        df = df.rename(columns=cols_to_rename)
        dataframes.append(df)
        
    master_df = dataframes[0]
    for next_df in dataframes[1:]:
        master_df = pd.merge_asof(master_df, next_df, on=time_col, by=id_col, direction='backward')
        
    return master_df.groupby(id_col).ffill().bfill()

def temporal_train_test_split(df: pd.DataFrame, time_col: str, train_ratio: float = 0.8) -> tuple:
    df_sorted = df.sort_values(by=time_col).reset_index(drop=True)
    split_idx = int(len(df_sorted) * train_ratio)
    return df_sorted.iloc[:split_idx].copy(), df_sorted.iloc[split_idx:].copy()

def data_resilience_adapter(df: pd.DataFrame, feature_cols: list, window: int = 5, lags: int = 3) -> pd.DataFrame:
    df_engineered = df.copy()
    for col in feature_cols:
        df_engineered[f'{col}_roll_mean'] = df_engineered[col].rolling(window=window).mean()
        df_engineered[f'{col}_roll_std'] = df_engineered[col].rolling(window=window).std()
        df_engineered[f'{col}_roll_min'] = df_engineered[col].rolling(window=window).min()
        df_engineered[f'{col}_roll_max'] = df_engineered[col].rolling(window=window).max()
        for k in range(1, lags + 1):
            df_engineered[f'{col}_lag_{k}'] = df_engineered[col].shift(k)
    return df_engineered.dropna()

## 2.5 Verification
To verify if any curveballs are thrown

In [ ]:
import pandas as pd
import numpy as np

def run_hackathon_diagnostics(train_df: pd.DataFrame, test_df: pd.DataFrame, id_col: str, target_fault_col: str, target_sensor_col: str):
    """
    Scans the data for the 5 hackathon curveballs and prints a health report.
    """
    print("==================================================")
    print("🚨 HACKATHON DAY 1 DIAGNOSTIC RADAR 🚨")
    print("==================================================\n")
    
    # ---------------------------------------------------------
    # Curveball 1 & 5: Extreme Class Imbalance & Contamination
    # ---------------------------------------------------------
    fault_ratio = train_df[target_fault_col].mean()
    print("1. CLASS IMBALANCE & CONTAMINATION CHECK")
    print(f"   -> True Fault Rate in Training Data: {fault_ratio:.4%} ({train_df[target_fault_col].sum()} faults out of {len(train_df)} rows)")
    
    if fault_ratio < 0.01:
        print("   ⚠️ WARNING: Extreme class imbalance detected (<1%).")
        print("   💡 FIX: Add `scale_pos_weight` to Model 2 (XGBClassifier).")
    elif fault_ratio > 0.10:
        print("   ⚠️ WARNING: High contamination detected (>10%).")
        print(f"   💡 FIX: Update Model 1 (IsolationForest) `contamination={fault_ratio:.3f}`.")
    else:
        print("   ✅ OK: Fault ratio is balanced for standard parameters.")
    print("-" * 50)

    # ---------------------------------------------------------
    # Curveball 2: The "New Train" Identity Crisis
    # ---------------------------------------------------------
    train_ids = set(train_df[id_col].unique())
    test_ids = set(test_df[id_col].unique())
    missing_in_train = test_ids - train_ids
    
    print("2. CHRONOLOGICAL SPLIT LEAKAGE (TRAIN ID CHECK)")
    print(f"   -> Unique Trains in Train Set: {len(train_ids)}")
    print(f"   -> Unique Trains in Test Set:  {len(test_ids)}")
    
    if missing_in_train:
        print(f"   🚨 CRITICAL: Test set contains unseen trains: {missing_in_train}")
        print("   💡 FIX: The time split isolated new trains. You must group your StandardScaling by Train ID, or train separate models per ID.")
    else:
        print("   ✅ OK: All trains in the test set were seen during training.")
    print("-" * 50)

    # ---------------------------------------------------------
    # Curveball 3: The Extrapolation Wall
    # ---------------------------------------------------------
    train_max = train_df[target_sensor_col].max()
    test_max = test_df[target_sensor_col].max()
    
    print(f"3. EXTRAPOLATION WALL (TARGET: {target_sensor_col})")
    print(f"   -> Train Max Value: {train_max:.2f}")
    print(f"   -> Test Max Value:  {test_max:.2f}")
    
    if test_max > (train_max * 1.1):
        print(f"   🚨 CRITICAL: Test set peaks much higher than train set (+10%).")
        print("   💡 FIX: Model 3 (XGBRegressor) will flatline at the train max. Switch Model 3 to a linear regressor (e.g., Ridge).")
    else:
        print("   ✅ OK: Test target values fall within historical bounds.")
    print("-" * 50)

    # ---------------------------------------------------------
    # Curveball 4: Flatline Forward-Fill (Zero Variance Features)
    # ---------------------------------------------------------
    print("4. FLATLINE SENSORS (ZERO VARIANCE CHECK)")
    # Grab only numeric feature columns (ignore IDs and timestamps)
    numeric_cols = train_df.select_dtypes(include=[np.number]).columns
    zero_variance_cols = [col for col in numeric_cols if train_df[col].std() == 0]
    
    if zero_variance_cols:
        print(f"   ⚠️ WARNING: Found {len(zero_variance_cols)} columns with zero variance (completely flat).")
        print(f"   -> {zero_variance_cols[:5]} ...")
        print("   💡 FIX: `merge_asof` ffill likely dragged a low-frequency sensor across millions of rows. Drop these features or aggregate the main dataset to a lower frequency.")
    else:
        print("   ✅ OK: All numeric sensors show variance.")
    print("==================================================\n")

# Example Execution for Block 2.5:
# run_hackathon_diagnostics(
#     train_df=train_df, 
#     test_df=test_df, 
#     id_col='train_id', 
#     target_fault_col='faults_fault_present', 
#     target_sensor_col='bogie_temperature'
# )

## 3. Core AI Models
Definitions for the Anomaly Detector (Model 1), Supervised Classifier (Model 2), and Temporal Forecaster (Model 3).

In [ ]:
# === MODEL 1: ANOMALY DETECTOR ===
def train_anomaly_detector(df: pd.DataFrame, features: list) -> pd.DataFrame:
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df[features])
    iso_forest = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
    iso_forest.fit(X_scaled)
    df['anomaly_label'] = iso_forest.predict(X_scaled)
    df['anomaly_risk_score'] = iso_forest.score_samples(X_scaled) * -1 
    return df

# === MODEL 2: SUPERVISED CLASSIFIER ===
def train_supervised_classifier(X_train: pd.DataFrame, y_train: pd.Series, X_test: pd.DataFrame) -> tuple:
    xgb_model = xgb.XGBClassifier(objective='binary:logistic', n_estimators=150, learning_rate=0.1, max_depth=5, random_state=42, eval_metric='logloss')
    xgb_model.fit(X_train, y_train)
    fault_probabilities = xgb_model.predict_proba(X_test)[:, 1]
    feature_importance = pd.DataFrame({'Feature': X_train.columns, 'Importance': xgb_model.feature_importances_}).sort_values(by='Importance', ascending=False)
    return xgb_model, fault_probabilities, feature_importance

def generate_shap_rankings(model, X_test: pd.DataFrame):
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)
    return explainer, shap_values

# === MODEL 3: TEMPORAL FORECASTER ===
def create_forecasting_target(df: pd.DataFrame, target_col: str, horizon: int = 5) -> pd.DataFrame:
    df_forecasting = df.copy()
    df_forecasting[f'{target_col}_future_{horizon}'] = df_forecasting[target_col].shift(-horizon)
    return df_forecasting.dropna()

def train_temporal_forecaster(X_train: pd.DataFrame, y_train: pd.Series, X_test: pd.DataFrame) -> tuple:
    xgb_regressor = xgb.XGBRegressor(n_estimators=150, learning_rate=0.05, max_depth=6, random_state=42, objective='reg:squarederror')
    xgb_regressor.fit(X_train, y_train)
    return xgb_regressor, xgb_regressor.predict(X_test)

def detect_future_breaches(predicted_values: np.ndarray, upper_threshold: float) -> np.ndarray:
    return (predicted_values >= upper_threshold).astype(int)

## 4. Master Pipeline Execution
The wrapper function that runs the entire workflow end-to-end.

In [ ]:
def run_full_ai_pipeline(csv_file_dict: dict, time_col: str, id_col: str, target_fault_col: str, target_sensor_col: str) -> dict:
    print("--- 1. Ingesting and Merging Data ---")
    master_df = merge_telemetry_datasets(csv_file_dict, time_col, id_col)
    print(f"Master dataset merged. Total shape: {master_df.shape}")
    
    print("\n--- 2. Splitting Data ---")
    train_df, test_df = temporal_train_test_split(master_df, time_col, train_ratio=0.8)
    feature_cols = [col for col in master_df.columns if col not in [time_col, id_col, target_fault_col]]
    
    print("\n--- 3. Running Data Resilience Adapter ---")
    train_eng = data_resilience_adapter(train_df, feature_cols, window=5, lags=3)
    test_eng = data_resilience_adapter(test_df, feature_cols, window=5, lags=3)
    
    X_train = train_eng[feature_cols]
    y_train = train_eng[target_fault_col]
    X_test = test_eng[feature_cols]
    
    print("\n--- 4. Executing Model 1: Anomaly Detector ---")
    test_with_anomalies = train_anomaly_detector(test_eng.copy(), feature_cols)
    
    print("\n--- 5. Executing Model 2: Supervised Classifier ---")
    xgb_classifier, fault_probs, feature_importance = train_supervised_classifier(X_train, y_train, X_test)
    explainer, shap_vals = generate_shap_rankings(xgb_classifier, X_test)
    
    print("\n--- 6. Executing Model 3: Temporal Forecaster ---")
    train_forecast = create_forecasting_target(train_eng, target_sensor_col, horizon=5)
    X_train_forecast = train_forecast[feature_cols]
    y_train_forecast = train_forecast[f'{target_sensor_col}_future_5']
    xgb_regressor, future_sensor_predictions = train_temporal_forecaster(X_train_forecast, y_train_forecast, X_test)
    breach_alerts = detect_future_breaches(future_sensor_predictions, upper_threshold=85.0)
    
    print("\n--- Pipeline Execution Finished ---")
    return {
        'test_data': test_eng,
        'anomaly_scores': test_with_anomalies['anomaly_risk_score'],
        'fault_probabilities': fault_probs,
        'shap_values': shap_vals,
        'future_predictions': future_sensor_predictions,
        'breach_alerts': breach_alerts
    }

## 5. Submission Formatter
Exports outputs to the exact *_predictions.csv layout required by LTA.

In [ ]:
def export_predictions_to_csv(results_dict: dict, test_df: pd.DataFrame, output_dir: str = "./submission_folder"):
    os.makedirs(output_dir, exist_ok=True)
    
    pd.DataFrame({'timestamp': test_df['timestamp'], 'train_id': test_df['train_id'], 'anomaly_risk_score': results_dict['anomaly_scores']}).to_csv(os.path.join(output_dir, "anomaly_predictions.csv"), index=False)
    pd.DataFrame({'timestamp': test_df['timestamp'], 'train_id': test_df['train_id'], 'fault_probability': results_dict['fault_probabilities']}).to_csv(os.path.join(output_dir, "fault_predictions.csv"), index=False)
    pd.DataFrame({'timestamp': test_df['timestamp'], 'train_id': test_df['train_id'], 'predicted_sensor_value': results_dict['future_predictions'], 'breach_alert_flag': results_dict['breach_alerts']}).to_csv(os.path.join(output_dir, "forecast_predictions.csv"), index=False)
    
    print(f"Success! All predictions exported to {output_dir}/")

## 6. Execution Cell
Uncomment and run this cell on Hackathon Day 1 once the dataset drops.

In [ ]:
# loaded_csvs = load_local_csvs("./csvs")

# pipeline_results = run_full_ai_pipeline(
#     csv_file_dict=loaded_csvs,
#     time_col='timestamp',
#     id_col='train_id',
#     target_fault_col='faults_fault_present', 
#     target_sensor_col='bogie_temperature'
# )

# export_predictions_to_csv(
#     results_dict=pipeline_results, 
#     test_df=pipeline_results['test_data'], 
#     output_dir="./LTA_PS3_Submission"
# )